# Audio Feature Extraction - Full Feature Set

**Extract 89 audio features from downloaded MP3 files**

**Features extracted:**
- Tempo (1)
- RMS Energy (2)
- Zero Crossing Rate (2)
- Spectral Centroid (2)
- Spectral Rolloff (2)
- Spectral Bandwidth (2)
- Spectral Contrast 7 bands (14)
- MFCC 13 coefficients (26)
- Chroma 12 pitch classes (24)
- Tonnetz 6 features (12)
- Onset Strength (2)

**Total: 89 features**

**Estimated time:** 2.4 hours with 12 workers for 45,790 songs

## 1. Setup & Imports

In [ ]:
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")
print(f"librosa version: {librosa.__version__}")

## 2. Configuration

In [ ]:
# Configuration
NUM_WORKERS = 16  # For your 16-core CPU
BATCH_SIZE = 500  # Save progress every 500 songs

# Paths
AUDIO_DIR = Path("../data/raw/youtube_audio")
OUTPUT_CSV = Path("../data/raw/audio_features.csv")

print(f"Workers: {NUM_WORKERS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Audio directory: {AUDIO_DIR}")
print(f"Output CSV: {OUTPUT_CSV}")

## 3. Load Audio Files

In [ ]:
# Get all MP3 files
audio_files = list(AUDIO_DIR.glob("*.mp3"))

print(f"Total audio files found: {len(audio_files):,}")

if len(audio_files) == 0:
    print("\n⚠️ WARNING: No audio files found!")
    print(f"   Check that {AUDIO_DIR} contains .mp3 files")

## 4. Check for Existing Progress

In [ ]:
# Check if we have existing progress
if OUTPUT_CSV.exists():
    existing_features = pd.read_csv(OUTPUT_CSV)
    already_processed = set(existing_features['track_id'].values)
    
    print(f"Found existing features file")
    print(f"Already processed: {len(already_processed):,} tracks")
    
    # Filter to unprocessed files
    pending_files = [
        f for f in audio_files 
        if f.stem not in already_processed
    ]
    
    print(f"Remaining to process: {len(pending_files):,} tracks")
else:
    print("No existing features file found - starting fresh")
    pending_files = audio_files
    existing_features = None

print(f"\nTotal files to process: {len(pending_files):,}")

## 5. Feature Extraction Function

In [ ]:
def extract_full_features(audio_path):
    """
    Extract complete 89-feature set from audio file
    
    Features:
    - Tempo (1)
    - RMS Energy (2: mean, std)
    - Zero Crossing Rate (2: mean, std)
    - Spectral Centroid (2: mean, std)
    - Spectral Rolloff (2: mean, std)
    - Spectral Bandwidth (2: mean, std)
    - Spectral Contrast 7 bands (14: 7 means, 7 stds)
    - MFCC 13 coefficients (26: 13 means, 13 stds)
    - Chroma 12 pitch classes (24: 12 means, 12 stds)
    - Tonnetz 6 features (12: 6 means, 6 stds)
    - Onset Strength (2: mean, std)
    
    Total: 89 features
    """
    try:
        # Extract track_id from filename
        track_id = audio_path.stem
        
        # Load audio (max 3 minutes)
        y, sr = librosa.load(str(audio_path), sr=22050, mono=True, duration=180)
        
        features = {'track_id': track_id}
        
        # 1. Tempo (1 feature)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        if isinstance(tempo, np.ndarray):
            tempo = tempo[0] if len(tempo) > 0 else 0.0
        features['tempo'] = float(tempo)
        
        # 2. RMS Energy (2 features)
        rms = librosa.feature.rms(y=y)
        features['rms_mean'] = float(np.mean(rms))
        features['rms_std'] = float(np.std(rms))
        
        # 3. Zero Crossing Rate (2 features)
        zcr = librosa.feature.zero_crossing_rate(y)
        features['zcr_mean'] = float(np.mean(zcr))
        features['zcr_std'] = float(np.std(zcr))
        
        # 4. Spectral Centroid (2 features)
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)
        features['spectral_centroid_mean'] = float(np.mean(spectral_centroids))
        features['spectral_centroid_std'] = float(np.std(spectral_centroids))
        
        # 5. Spectral Rolloff (2 features)
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features['spectral_rolloff_mean'] = float(np.mean(spectral_rolloff))
        features['spectral_rolloff_std'] = float(np.std(spectral_rolloff))
        
        # 6. Spectral Bandwidth (2 features)
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features['spectral_bandwidth_mean'] = float(np.mean(spectral_bandwidth))
        features['spectral_bandwidth_std'] = float(np.std(spectral_bandwidth))
        
        # 7. Spectral Contrast (14 features: 7 bands × 2)
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        for i in range(7):
            features[f'spectral_contrast_{i+1}_mean'] = float(np.mean(spectral_contrast[i]))
            features[f'spectral_contrast_{i+1}_std'] = float(np.std(spectral_contrast[i]))
        
        # 8. MFCC (26 features: 13 coefficients × 2)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(13):
            features[f'mfcc_{i+1}_mean'] = float(np.mean(mfccs[i]))
            features[f'mfcc_{i+1}_std'] = float(np.std(mfccs[i]))
        
        # 9. Chroma (24 features: 12 pitch classes × 2)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            features[f'chroma_{i+1}_mean'] = float(np.mean(chroma[i]))
            features[f'chroma_{i+1}_std'] = float(np.std(chroma[i]))
        
        # 10. Tonnetz (12 features: 6 tonal features × 2)
        tonnetz = librosa.feature.tonnetz(y=y, sr=sr)
        for i in range(6):
            features[f'tonnetz_{i+1}_mean'] = float(np.mean(tonnetz[i]))
            features[f'tonnetz_{i+1}_std'] = float(np.std(tonnetz[i]))
        
        # 11. Onset Strength (2 features)
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        features['onset_strength_mean'] = float(np.mean(onset_env))
        features['onset_strength_std'] = float(np.std(onset_env))
        
        features['extraction_success'] = True
        features['error_message'] = ''
        
        return features
        
    except Exception as e:
        return {
            'track_id': audio_path.stem,
            'extraction_success': False,
            'error_message': str(e)
        }

print("✓ Feature extraction function defined")
print("  Expected features per song: 89")

## 6. Test Extraction on Single File

In [ ]:
# Test on single file before processing all
if pending_files:
    test_file = pending_files[0]
    
    print(f"Testing extraction on: {test_file.name}")
    print("\nExtracting features...")
    
    start_time = time.time()
    test_features = extract_full_features(test_file)
    elapsed = time.time() - start_time
    
    if test_features.get('extraction_success'):
        feature_count = len([k for k in test_features.keys() if k not in ['track_id', 'extraction_success', 'error_message']])
        
        print(f"\n✓ Extraction successful!")
        print(f"  Features extracted: {feature_count}")
        print(f"  Time: {elapsed:.2f} seconds")
        print(f"\n  Sample features:")
        print(f"    Tempo: {test_features.get('tempo', 'N/A'):.2f} BPM")
        print(f"    RMS Mean: {test_features.get('rms_mean', 'N/A'):.6f}")
        print(f"    Spectral Centroid Mean: {test_features.get('spectral_centroid_mean', 'N/A'):.2f} Hz")
        
        # Projection
        total_time_hours = (len(pending_files) * elapsed) / (NUM_WORKERS * 3600)
        print(f"\n  Projected time for {len(pending_files):,} songs with {NUM_WORKERS} workers:")
        print(f"    {total_time_hours:.1f} hours ({total_time_hours/24:.1f} days)")
    else:
        print(f"\n✗ Extraction failed!")
        print(f"  Error: {test_features.get('error_message', 'Unknown error')}")
else:
    print("No files to process!")

## 7. Parallel Feature Extraction

In [ ]:
def process_batch(files, num_workers=12, batch_size=500):
    """
    Process files in batches with parallel workers
    """
    all_features = []
    
    # Load existing if available
    if OUTPUT_CSV.exists():
        existing = pd.read_csv(OUTPUT_CSV)
        all_features = existing.to_dict('records')
        print(f"Loaded {len(all_features):,} existing features\n")
    
    total_files = len(files)
    
    # Process in batches
    for batch_start in range(0, total_files, batch_size):
        batch_end = min(batch_start + batch_size, total_files)
        batch_files = files[batch_start:batch_end]
        
        print(f"\n{'='*70}")
        print(f"Batch: {batch_start + 1}-{batch_end} of {total_files:,}")
        print(f"Workers: {num_workers}")
        print(f"{'='*70}\n")
        
        batch_features = []
        
        # Use ProcessPoolExecutor for CPU-bound work
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            # Submit all tasks
            futures = {
                executor.submit(extract_full_features, file): file 
                for file in batch_files
            }
            
            # Collect results with progress bar
            with tqdm(total=len(batch_files), desc=f"Batch {batch_start//batch_size + 1}") as pbar:
                for future in as_completed(futures):
                    try:
                        features = future.result()
                        batch_features.append(features)
                    except Exception as e:
                        file = futures[future]
                        batch_features.append({
                            'track_id': file.stem,
                            'extraction_success': False,
                            'error_message': f'Worker error: {str(e)}'
                        })
                    finally:
                        pbar.update(1)
        
        # Add to all features
        all_features.extend(batch_features)
        
        # Save progress
        features_df = pd.DataFrame(all_features)
        features_df.to_csv(OUTPUT_CSV, index=False)
        
        # Show batch stats
        batch_success = sum(f.get('extraction_success', False) for f in batch_features)
        
        print(f"\n✓ Batch complete!")
        print(f"  Total progress: {len(all_features):,} songs")
        print(f"  This batch - Success: {batch_success}/{len(batch_features)}")
        print(f"  Saved to: {OUTPUT_CSV}")
    
    return pd.DataFrame(all_features)

# Run extraction
print("Starting feature extraction...\n")
print(f"Processing {len(pending_files):,} files with {NUM_WORKERS} workers\n")

features_df = process_batch(
    pending_files,
    num_workers=NUM_WORKERS,
    batch_size=BATCH_SIZE
)

print(f"\n{'='*70}")
print("FEATURE EXTRACTION COMPLETE!")
print(f"{'='*70}")

## 8. Analyze Extracted Features

In [ ]:
# Load and analyze features
if OUTPUT_CSV.exists():
    features_df = pd.read_csv(OUTPUT_CSV)
    
    print("="*70)
    print("FEATURE EXTRACTION SUMMARY")
    print("="*70)
    
    print(f"\nTotal songs processed: {len(features_df):,}")
    
    if 'extraction_success' in features_df.columns:
        successful = features_df['extraction_success'].sum()
        failed = (~features_df['extraction_success']).sum()
        
        print(f"Successful: {successful:,} ({(successful/len(features_df)*100):.1f}%)")
        print(f"Failed: {failed:,} ({(failed/len(features_df)*100):.1f}%)")
    
    # Get feature columns (exclude metadata)
    feature_cols = [col for col in features_df.columns 
                    if col not in ['track_id', 'extraction_success', 'error_message']]
    
    print(f"\nFeatures per song: {len(feature_cols)}")
    
    # Summary statistics
    if len(feature_cols) > 0:
        print(f"\nFeature Summary (first 10 features):")
        print(features_df[feature_cols[:10]].describe().T[['mean', 'std', 'min', 'max']])
    
    print(f"\n✓ Features saved to: {OUTPUT_CSV}")
else:
    print("No features file found yet - run extraction first!")

## 9. Merge with Spotify Metadata (Optional)

In [ ]:
# Optionally merge audio features with Spotify metadata
# spotify_df = pd.read_csv("../data/raw/spotify_tracks.csv")
# results_df = pd.read_csv("../data/raw/combined_match_download_results.csv")
# features_df = pd.read_csv(OUTPUT_CSV)

# # Merge all datasets
# complete_df = spotify_df.merge(results_df, on='track_id', how='left')
# complete_df = complete_df.merge(features_df, on='track_id', how='left')

# # Save complete dataset
# complete_df.to_csv("../data/raw/complete_dataset_with_features.csv", index=False)

# print(f"✓ Complete dataset saved!")
# print(f"  Total records: {len(complete_df):,}")
# print(f"  Total columns: {len(complete_df.columns)}")

## Next Steps

**After feature extraction completes:**

1. ✅ Audio features extracted (89 per song)
2. ⏳ Collect YouTube metadata (views, likes, etc.)
3. ⏳ Merge all datasets
4. ⏳ Build virality prediction model

**Files created:**
- `audio_features.csv` - Audio features for all songs
- `complete_dataset_with_features.csv` - Combined Spotify + YouTube + Audio features